In [ ]:
import os
import ee
import geemap
import geopandas as gpd
from pathlib import Path

In [ ]:
# Initialize Earth Engine with your active project
# Explicit Google Cloud Project ID
PROJECT_ID = "weighty-arcadia-488508-k8"

try:
    ee.Initialize(project=PROJECT_ID)
    print(f"Earth Engine successfully initialized with project: {PROJECT_ID}")
except Exception as e:
    print("Initial attempt failed, triggering re-authentication...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine authenticated and initialized successfully.")

In [ ]:
Map = geemap.Map()

In [ ]:

# Load Boundary via Dynamic Relative Path (Native EPSG:4326)
BASE_DIR = Path.cwd()
# If executing from inside the notebooks/ folder, step back to project root
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

bnd_path = os.path.join(
    PROJECT_ROOT, "data", "raw", "ihr_boundary_dissolved.gpkg"
)

# Load layer directly (native EPSG:4326 verified)
gdf_bnd = gpd.read_file(bnd_path)

# Convert GeoPandas geometry directly to Earth Engine FeatureCollection/Geometry
ihr_fc = geemap.gdf_to_ee(gdf_bnd)
ihr_geom = ihr_fc.geometry()

In [ ]:

# Latest available Hansen product covering gross loss events up to 2025
gfc_asset_id = "UMD/hansen/global_forest_change_2025_v1_13"
gfc = ee.Image(gfc_asset_id).clip(ihr_geom)

# Print band details and metadata
band_names = gfc.bandNames().getInfo()
print(f"Hansen GFC Asset Loaded: {gfc_asset_id}")
print(f"Available Bands: {band_names}")

In [ ]:

# 4. Interactive Visualisation via geemap (2001-2025)
Map = geemap.Map()
Map.centerObject(ihr_fc, zoom=6)

# 1. Base Boundary Outline
Map.addLayer(
    ihr_fc, {"color": "black", "fillColor": "00000000", "width": 2}, "IHR Boundary"
)

# 2. Baseline Tree Canopy Cover in Year 2000
treecover = gfc.select("treecover2000")
treecover_vis = {
    "min": 0,
    "max": 100,
    "palette": ["#ffffff", "#a1d99b", "#238b45", "#00441b"],
}
Map.addLayer(treecover, treecover_vis, "Tree Cover 2000 (%)", False)

# 3. Baseline Canopy Mask (>25%) as defined in methodology
baseline_mask = treecover.gte(25).updateMask(gfc.select("datamask").eq(1))
Map.addLayer(
    baseline_mask.selfMask(),
    {"palette": ["#2e7d32"]},
    "Baseline Forest Mask (>25%)",
    False,
)

# 4. Gross Forest Loss (2001-2025)
loss = gfc.select("loss")
loss_vis = {"palette": ["#d73027"]}
Map.addLayer(
    loss.selfMask(), loss_vis, "Gross Forest Loss (2001-2025)", True
)

# 5. Spatiotemporal Loss Progression (Years 1 to 25)
lossyear = gfc.select("lossyear")
lossyear_vis = {
    "min": 1,
    "max": 25,
    "palette": [
        "#ffffcc",  # Early (2001-2005) --> Light Yellow to Deep Green
        "#c2e699",
        "#78c679",
        "#31a354",
        "#006837",
        "#fed976",  # Mid (2006-2015) --> Light Orange to Dark Red
        "#fd8d3c",
        "#f03b20",
        "#bd0026",
        "#7a0177",  # Recent (2016-2025) --> Deep magenta/purple to Eggplant colour
        "#49006a",
    ],
}
Map.addLayer(
    lossyear.selfMask(),
    lossyear_vis,
    "Loss Year (1=2001 to 25=2025)",
    False,
)

Map